# Phase 2 — laws_de.csv Hybrid Retrieval (BM25 + BGE-M3 + RRF)

**Goal**: dense + sparse retrieval over Swiss federal laws. court_considerations skipped (Phase 3).

**Expected LB**: 0.20~0.30.

**Settings (set BEFORE running)**:
- Accelerator: **GPU T4×2**
- Internet: **ON** for first run (BGE-M3 download). After cache: OFF works.
- Persistence: ON (FAISS/BM25 indexes cached to /kaggle/working)

Pipeline per query:
1. Regex seeds (Art./BGE/BGer mentions, multilang-aware) + universal defaults
2. BM25 search over laws_de.text
3. BGE-M3 dense over laws_de.text
4. RRF fusion → top-K
5. Granularity filter
6. Emit ~20 citations

In [ ]:
# Cell 1 — pip + paths (auto-detect data location)
!pip install -q rank_bm25 sentence-transformers faiss-cpu

import re, csv, json, pickle, sys
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

for cand in [
    Path('/kaggle/input/llm-agentic-legal-information-retrieval'),
    Path('/kaggle/input/competitions/llm-agentic-legal-information-retrieval'),
]:
    if cand.exists() and (cand / 'laws_de.csv').exists():
        DATA = cand
        break
else:
    raise RuntimeError(f"Data not found. /kaggle/input contents: {list(Path('/kaggle/input').iterdir())}")
print(f'DATA: {DATA}')

WORK = Path('/kaggle/working')
CACHE = WORK / 'cache_phase2'
CACHE.mkdir(parents=True, exist_ok=True)
for p in ['train.csv', 'val.csv', 'test.csv', 'laws_de.csv']:
    assert (DATA / p).exists(), f'missing: {DATA / p}'
print('files OK')

In [ ]:
import json

# Cell 2 — Multilingual abbreviation map (FR/IT → DE) — 1662 entries from Omnilex starter repo
MULTILANG_ABBR = {'OIAA': 'AAMV', 'OMAA': 'HVUV', 'DE-OMBat': 'AB-VASm', 'OBE-FINMA': 'ABV-FINMA', 'OAdo': 'AdoV', 'OAdoz': 'AdoV', 'SDA': 'ADS', 'ORAT': 'AEFV', 'OIAgr': 'AEV', 'LMCFA': 'AFZFG', 'LMCCE': 'AFZFG', 'OMCFA': 'AFZFV', 'OMCCE': 'AFZFV', 'LAVS': 'AHVG', 'RAVS': 'AHVV', 'OAVS': 'AHVV', 'LEAR': 'AIAG', 'LSAI': 'AIAG', 'OEAR': 'AIAV', 'OSAIn': 'AIAV', 'LEI': 'AIG', 'LStrI': 'AIG', 'OAccD': 'AkkBV', 'AccredO-LPsy': 'AkkredV-PsyG', 'OEAc-LPPsi': 'AkkredV-PsyG', 'LEDPP': 'ALBAG', 'LSRPP': 'ALBAG', 'OEDPP': 'ALBAV', 'OSRPP': 'ALBAV', 'OBat': 'AlgV', 'OSRA': 'V-NDA', 'OdA': 'AlkBestV', 'OTAl': 'AlkBestV', 'LAlc': 'AlkG', 'OAlc': 'AlkV', 'OAllerg': 'AllergV', 'OGEmol': 'AllgGebV', 'OgeEm': 'AllgGebV', 'OSites': 'AltlV', 'OSiti': 'AltlV', 'OSI-AC': 'ALV-IsV', 'OAMéd': 'AMBV', 'OAMed': 'AMBV', 'OEMéd': 'AMZV', 'OOMed': 'AMZV', 'OOrgA': 'AO', 'OEs': 'AO', 'OOS': 'AOV', 'OOV': 'AOV', 'LTr': 'ArG', 'LL': 'ArG', 'OLT 1': 'ArGV 1', 'OLL 1': 'ArGV 1', 'OLT 2': 'ArGV 2', 'OLL 2': 'ArGV 2', 'OLT 3': 'ArGV 3', 'OLL 3': 'ArGV 3', 'OLT 4': 'ArGV 4', 'OLL 4': 'ArGV 4', 'OLT 5': 'ArGV 5', 'OLL 5': 'ArGV 5', 'OITRV': 'ARPV', 'OTR 1': 'ARV 1', 'OLR 1': 'ARV 1', 'OTR 2': 'ARV 2', 'OLR 2': 'ARV 2', 'LSEtr': 'ASG', 'LSEst': 'ASG', 'Limpauto': 'AStG', 'LIAut': 'AStG', 'Oimpauto': 'AStV', 'OIAut': 'AStV', 'OSur-ASR': 'ASV-RAB', 'OS-ASR': 'ASV-RAB', 'LAsi': 'AsylG', 'OA 1': 'AsylV 1', 'OAsi 1': 'AsylV 1', 'OA 2': 'AsylV 2', 'OAsi 2': 'AsylV 2', 'OA 3': 'AsylV 3', 'OAsi 3': 'AsylV 3', 'LTrAlp': 'AtraG', 'LTAlp': 'AtraG', 'Otransa': 'AtraV', 'OTrAl': 'AtraV', 'LPGA': 'ATSG', 'OPGA': 'ATSV', 'RSTF': 'AufRBGer', 'RVTF': 'AufRBGer', 'OAsc': 'AufzV', 'OSAC': 'AuLaV', 'OAEs': 'AuLaV', 'LECCT': 'AVEG', 'LOCCL': 'AVEG', 'OFAC': 'AVFV', 'OFAD': 'AVFV', 'LSE': 'AVG', 'LC': 'AVG', 'LACI': 'AVIG', 'LADI': 'AVIG', 'OACI': 'AVIV', 'OADI': 'AVIV', 'OS': 'AVO', 'OS-FINMA': 'AVO-FINMA', 'OSE': 'AVV', 'OC': 'AVV', 'LDI': 'AwG', 'LDT': 'AZG', 'LDL': 'AZG', 'OLDT': 'AZGV', 'OLDL': 'AZGV', 'ODMA': 'BAlV', 'LB': 'BankG', 'LBCR': 'BankG', 'OB': 'BankV', 'OBCR': 'BankV', 'OTConst': 'BauAV', 'OLCostr': 'BauAV', 'LPCo': 'BauPG', 'LProdC': 'BauPG', 'OPCo': 'BauPV', 'OProdC': 'BauPV', 'LFPr': 'BBG', 'OFPr': 'BBV', 'LTI': 'BEG', 'LTCo': 'BEG', 'LHand': 'BehiG', 'LDis': 'BehiG', 'OHand': 'BehiV', 'ODis': 'BehiV', 'ONo-ASR': 'BekV-RAB', 'OC-ASR': 'BekV-RAB', 'LStup': 'BetmG', 'OCStup': 'BetmKV', 'OEPStup': 'BetmPV', 'OSPStup': 'BetmPV', 'OAStup': 'BetmSV', 'ODStup': 'BetmSV', 'OTStup-DFI': 'BetmVV-EDI', 'OEStup-DFI': 'BetmVV-EDI', 'OProP': 'BevSV', 'OPPop': 'BevSV', 'LFAIE': 'BewG', 'LAFE': 'BewG', 'OAIE': 'BewV', 'OAFE': 'BewV', 'LF-CLaH': 'BG-HAÜ', 'LF-CAA': 'BG-HAÜ', 'LTEJUS': 'BG-RVUS', 'LTAGSU': 'BG-RVUS', 'LAr': 'BGA', 'LDFR': 'BGBB', 'LMI': 'BGBM', 'LCITES': 'BGCITES', 'LF-CITES': 'BGCITES', 'RTF': 'BGerR', 'LFSP': 'BGF', 'LLCA': 'BGFA', 'LTF': 'BGG', 'LDEA': 'BGIAA', 'LSISA': 'BGIAA', 'LBCF': 'BGLE', 'LRFF': 'BGLE', 'LPPS': 'BGMD', 'LDPS': 'BGMD', 'LFPC': 'BGMK', 'LTrans': 'BGÖ', 'LTras': 'BGÖ', 'LEAC': 'BGRB', 'LIAC': 'BGRB', 'LJAr': 'BGS', 'LGD': 'BGS', 'LTN': 'BGSA', 'LLN': 'BGSA', 'LOST': 'BGST', 'LFSI': 'BGST', 'LFIF': 'BIFG', 'OBiG': 'BiGV', 'OIB-FINMA': 'BIV-FINMA', 'LCESF': 'BiZG', 'LCSFS': 'BiZG', 'LCMIF': 'BIZMB', 'OMPr': 'BMV', 'LMP': 'BöB', 'LAPub': 'BöB', 'OPDC': 'BPDV', 'OPDPers': 'BPDV', 'LSIP': 'BPI', 'LDP': 'BPR', 'LPSP': 'BPS', 'RNC': 'BSO', 'LSF': 'BStatG', 'LStat': 'BStatG', 'LIB': 'BStG', 'RAATPF': 'BStGerNR', 'ROATPF': 'BStGerNR', 'ROTPF': 'BStGerOR', 'RFPPF': 'BStKR', 'RSPPF': 'BStKR', 'OBioc': 'VBP', 'OBcarb': 'BTrV', 'LN': 'BüG', 'LCit': 'BüG', 'LSCPT': 'BÜPF', 'OREE': 'BURV', 'ORIS': 'BURV', 'OLN': 'VOSA', 'OCit': 'BüV', 'Cst.': 'BV', 'Cost.': 'BV', 'LPP': 'BVG', 'OPP 1': 'BVV 1', 'OPP 2': 'BVV 2', 'OPP 3': 'BVV 3', 'LIDV': 'BVVG', 'LDDV': 'BVVG', 'LMSI': 'BWIS', 'PCF': 'BZP', 'PC': 'BZP', 'OCart': 'CartV', 'LChim': 'ChemG', 'LPChim': 'ChemG', 'OEChim': 'ChemGebV', 'OEPChim': 'ChemGebV', 'OPICChim': 'ChemPICV', 'ORRChim': 'ChemRRV', 'ORRPChim': 'ChemRRV', 'OChim': 'ChemV', 'OPChim': 'ChemV', 'OCPCh': 'ChKV', 'OCCC': 'ChKV', 'CRTIF': 'COTIF 1980', 'LCaS-COVID-19': 'Covid-19-SBüG', 'LFiS-COVID-19': 'Covid-19-SBüG', 'OCyS': 'CSV', 'OCS': 'GGBV', 'CAC': 'CWÜ', 'OACP': 'CZV', 'OAut': 'CZV', 'ACCEES': 'ASEEGKVBSPMSA', 'ACSCSSS': 'ASEEGKVBSPMSA', 'LIFD': 'DBG', 'OSRP': 'DBV', 'OTDD': 'DBZV', 'LDes': 'DesG', 'ODes': 'DesV', 'OSEP': 'DGV', 'OSAP': 'DGV', 'RSA': 'DRA', 'RSE': 'DRA', 'LPD': 'DSG', 'OEng': 'DüV', 'OCon': 'DüV', 'OD-ASR': 'DV-RAB', 'OPD': 'DZV', 'LCdF': 'EBG', 'Lferr': 'EBG', 'OCF': 'EBV', 'Oferr': 'EBV', 'OITE-PT': 'EDAV-DS', 'OITE-PT-DFI': 'EDAV-DS-EDI', 'OITE-UE': 'EDAV-EU', 'OITE-UE-DFI': 'EDAV-EU-EDI', 'OITE-AC': 'EDAV-Ht', 'OITEAc': 'EDAV-Ht', 'OEES': 'EESV', 'O-HEFSM': 'EHSM-V', 'O-SUFSM': 'EHSM-V', 'OEmV': 'EichGebV', 'OEm-V': 'EichGebV', 'OIPR': 'EigVV', 'RIPP': 'EigVV', 'LIFM': 'EIMG', 'OIFM': 'EIMV', 'OO': 'EiV', 'OU': 'EiV', 'OCCP': 'EKBV', 'OCSC': 'EKBV', 'OESC-OFAG': 'EKV-BLW', 'OVC-UFAG': 'EKV-BLW', 'LIE': 'EleG', 'LPC': 'ELG', 'OPC-AVS/AI': 'ELV', 'LMETA': 'EMBAG', 'LMeCA': 'EMBAG', 'OMETA': 'EMBAV', 'OMeCA': 'EMBAV', 'LEmb': 'EmbG', 'OESMB': 'EmGvV', 'OACMB': 'EmGvV', 'LCMP': 'EMKG', 'OCMP': 'EMKV', 'OIMepe': 'EMmV', 'OSMisE': 'EMmV', 'CSDLLF': 'KSMG', 'CSDDDLF': 'KSMG', 'OEEE': 'EnEV', 'OEEne': 'EnEV', 'OEneR': 'EnFV', 'OPEn': 'EnFV', 'LIFSN': 'ENSIG', 'OIFSN': 'ENSIV', 'LDét': 'EntsG', 'LDist': 'EntsG', 'Odét': 'EntsV', 'ODist': 'EntsV', 'OAAE': 'EÖBV', 'OAPuE': 'EÖBV', 'OAAE-DFJP': 'EÖBV-EJPD', 'OAPuE-DFGP': 'EÖBV-EJPD', 'LAPG': 'EOG', 'LIPG': 'EOG', 'OAPG': 'EOV', 'OIPG': 'EOV', 'OFDEP': 'EPDFV', 'OFCIP': 'EPDFV', 'LDEP': 'EPDG', 'LCIP': 'EPDG', 'ODEP': 'EPDV', 'OCIP': 'EPDV', 'ODEP-DFI': 'EPDV-EDI', 'OCIP-DFI': 'EPDV-EDI', 'LEp': 'EpG', 'CBE 2000': 'EPÜ 2000', 'OEp': 'EpV', 'OFR': 'ERV', 'OFoP': 'ERV', 'CE-TAF': 'ERV-BVGer', 'OUC': 'ESV', 'OIConf': 'ESV', 'Oexpa': 'ExpaV', 'Oespa': 'ExpaV', 'LAFam': 'FamZG', 'OAFam': 'FamZV', 'OAFami': 'FamZV', 'OEV 4': 'FAV 4', 'OEA 4': 'FAV 4', 'OSVo': 'FDO', 'LSFin': 'FIDLEG', 'LSerFi': 'FIDLEG', 'OSFin': 'FIDLEV', 'OSerFi': 'FIDLEV', 'LERI': 'FIFG', 'LPRI': 'FIFG', 'OECin': 'FiFV', 'OPCin': 'FiFV', 'LIMF': 'FinfraG', 'LInFi': 'FinfraG', 'OIMF': 'FinfraV', 'OInFi': 'FinfraV', 'OIMF-FINMA': 'FinfraV-FINMA', 'OInFi-FINMA': 'FinfraV-FINMA', 'LEFin': 'FINIG', 'LIsFi': 'FINIG', 'OEFin': 'FINIV', 'OIsFi': 'FINIV', 'OEFin-FINMA': 'FINIV-FINMA', 'OIsFi-FINMA': 'FINIV-FINMA', 'Oém-FINMA': 'FINMA-GebV', 'Oem-FINMA': 'FINMA-GebV', 'OA-FINMA': 'FINMA-PV', 'LFINMA': 'FINMAG', 'OMPRI': 'FIPBV', 'LFiEl': 'FiREG', 'LAiSE': 'FiREG', 'CRSR': 'Abkommen\nüber die Rechtsstellung der Flüchtlinge', 'CSSR': 'Abkommen\nüber die Rechtsstellung der Flüchtlinge', 'LCF': 'FKG', 'OReCI': 'FKINV', 'OPRCI': 'FKINV', 'LFA': 'FLG', 'LAF': 'FLG', 'RFA': 'FLV', 'OA Fam': 'FLV', 'OLALA': 'FMBV', 'OLAlA': 'FMBV', 'LPMA': 'FMedG', 'LPAM': 'FMedG', 'OPMA': 'VLIp', 'OMP': 'VöB', 'LTC': 'FMG', 'OSALA': 'FMV', 'OsAlA': 'FMV', 'OFOrg': 'FOrgV', 'OAOrg': 'FOrgV', 'OH': 'FPV', 'OOra': 'FPV', 'OQICin': 'FQIV', 'OQIC': 'FQIV', 'ODE': 'FrSV', 'OEDA': 'FrSV', 'LFus': 'FusG', 'OMCo': 'FV', 'OMaeC': 'FV', 'OF-SCPT': 'FV-ÜPF', 'AECSDPC': 'ASEEGMF', 'ACSCS': 'ASEEGMF', 'OAG': 'GaGV', 'OAppG': 'GaGV', 'AGTDC': 'GATT', 'ORF': 'GBV', 'REmol-TAF': 'GebR-BVGer', 'TA-TAF': 'GebR-BVGer', 'REmol-TFB': 'GebR-PatGer', 'TA-TFB': 'GebR-PatGer', 'Olico': 'GeBüV', 'Olc': 'GeBüV', 'OE ArchF': 'GebV BAR', 'OE ARF': 'GebV BAR', 'OEmol-AFC': 'GebV ESTV', 'OEm-AFC': 'GebV ESTV', 'OELP': 'GebV SchKG', 'OTLEF': 'GebV SchKG', 'Oem-LEI': 'GebV-AIG', 'OEmol-LStrI': 'GebV-AIG', 'Oem-Acc': 'GebV-Akk', 'Oemo-Acc': 'GebV-Akk', 'OEmol-LTr': 'GebV-ArG', 'OEm-LL': 'GebV-ArG', 'OEmol-OFRO': 'GebV-ASTRA', 'OEmo-USTRA': 'GebV-ASTRA', 'OEmol-LSE': 'GebV-AVG', 'OEm-LC': 'GebV-AVG', 'OEmol-OFEV': 'GebV-BAFU', 'OE-UFAM': 'GebV-BAFU', 'OEmol-OFSPO': 'GebV-BASPO', 'OEm UFSPO': 'GebV-BASPO', 'OEmol-OFAC': 'GebV-BAZL', 'OEm-UFAC': 'GebV-BAZL', 'Oem-OFJ': 'GebV-BJ', 'Oem UFG': 'GebV-BJ', 'OEmol-OFAG': 'GebV-BLW', 'Ordinanza sulle tasse UFAG': 'GebV-BLW', 'OEmol-DFAE': 'GebV-EDA', 'OEm-DFAE': 'GebV-EDA', 'OEmol-DFI-BN': 'GebV-EDI-NBib', 'OEm-DFI-BN': 'GebV-EDI-NBib', 'OEmol-CMP': 'GebV-EMK', 'OEm-CMP': 'GebV-EMK', 'Oémol-En': 'GebV-En', 'OE-En': 'GebV-En', 'OEmol-ASF': 'GebV-ESA', 'OEm-AVF': 'GebV-ESA', 'OEmol-fedpol': 'GebV-fedpol', 'OEm-fedpol': 'GebV-fedpol', 'OREDT': 'GebV-FMG', 'OTST': 'GebV-FMG', 'OEmol-RC': 'GebV-HReg', 'OTa-IPI': 'GebV-IGE', 'OEmol-LCart': 'GebV-KG', 'OEm-LCart': 'GebV-KG', 'OEm-METAS': 'GebV-METAS', 'Oem-METAS': 'GebV-METAS', 'OEmol-BN': 'GebV-NBib', 'OEm-BN': 'GebV-NBib', 'OEmol-TP': 'GebV-öV', 'OEm-TP': 'GebV-öV', 'OEmol-Publ': 'GebV-Publ', 'OEm-Pub': 'GebV-Publ', 'OÉmol-CSA': 'GebV-SAR', 'OEm-CSA': 'GebV-SAR', 'OEmol-SEFRI': 'GebV-SBFI', 'OEm-SEFRI': 'GebV-SBFI', 'OE-RaP': 'GebV-StS', 'OEm-RaP': 'GebV-StS', 'OEmol-OHB': 'GebV-TPS', 'OEmSON': 'GebV-TPS', 'OEmol-DDPS': 'GebV-VBS', 'OEm-DDPS': 'GebV-VBS', 'OÉIPPSS': 'GebVO St', 'OSEIPSS': 'GebVO St', 'LGéo': 'GeoIG', 'LGI': 'GeoIG', 'OGéo': 'GeoIV', 'OGI': 'GeoIV', 'OGéo-swisstopo': 'GeoIV-swisstopo', 'OGI-swisstopo': 'GeoIV-swisstopo', 'OGéom': 'GeomV', 'Ogeom': 'GeomV', 'ONGéo': 'GeoNV', 'ONGeo': 'GeoNV', 'ORPSan': 'GesBAV', 'LPSan': 'GesBG', 'OCPSan': 'GesBKV', 'OSAS': 'GGBV', 'OCMD': 'GGUV', 'OMCont': 'GGUV', 'OIC-DFI': 'GgV-EDI', 'LCB': 'PAG', 'LBDI': 'GKG', 'ODVo': 'GKZ', 'OCPo': 'GKZ', 'LEg': 'GlG', 'LPar': 'GlG', 'OBPL': 'GLPV', 'RTFB': 'GR-PatGer', 'RI-COMCO': 'GR-WEKO', 'RCN': 'GRN', 'RCE': 'GRS', 'RCS': 'GRS', 'LEaux': 'GSchG', 'LPAc': 'GSchG', 'OEaux': 'GSchV', 'OPAc': 'GSchV', 'LGG': 'GTG', 'LIG': 'GTG', 'LAGH': 'GUMG', 'LEGU': 'GUMG', 'OAGH': 'GUMV', 'OEGU': 'GUMV', 'LTM': 'GüTG', 'OTM': 'GüTV', 'LTTM': 'GVVG', 'LTrasf': 'GVVG', 'LBA': 'GwG', 'LRD': 'GwG', 'OBA': 'GwV', 'ORD': 'GwV', 'OBA-OFDF': 'GwV-BAZG', 'ORD-UDSC': 'GwV-BAZG', 'OBA-DFJP': 'GwV-EJPD', 'ORD-DFGP': 'GwV-EJPD', 'OBA-CFMJ': 'GwV-ESBK', 'ORD-CFCG': 'GwV-ESBK', 'OBA-FINMA': 'GwV-FINMA', 'ORD-FINMA': 'GwV-FINMA', 'LTrD': 'HArG', 'LLD': 'HArG', 'OTrD': 'HArGV', 'OLLD': 'HArGV', 'OIPSD': 'HasLV', 'OIPSDA': 'HasLV', 'OIPSD-DEFR': 'HasLV-WBF', 'OIPSDA-DEFR': 'HasLV-WBF', 'OPFP-FINMA': 'HBEV-FINMA', 'LRH': 'HFG', 'LRUm': 'HFG', 'LEHE': 'HFKG', 'LPSU': 'HFKG', 'OMCR 20': 'HFMV 20', 'OPCR 20': 'HFMV 20', 'OMCR 22': 'HFMV 22', 'OPCR 22': 'HFMV 22', 'ORH': 'HFV', 'ORUm': 'HFV', 'LLGV': 'HGVAnG', 'LRAV': 'HGVAnG', 'OCBo': 'HHV', 'OCoL': 'HHV', 'CLaH96': 'HKsÜ', 'Convenzione dell’Aia sulla protezione dei minori': 'HKsÜ', 'OGOM': 'HKSV', 'OGOE': 'HKSV', 'LPTh': 'HMG', 'LATer': 'HMG', 'ORC': 'HRegV', 'OCCHE': 'HSBBV', 'OSCU': 'HSBBV', 'OMAV': 'HVA', 'OMAI': 'HVI', 'OMAINF': 'HVUV', 'OHyg': 'HyV', 'ORI': 'HyV', 'OIAM': 'IAMV', 'OSFPrHE': 'IBH-V', 'O-SIFPU': 'IBH-V', 'LSIS': 'SIaG', 'LSISpo': 'IBSG', 'OSIS': 'IBSV', 'OSISpo': 'IBSV', 'OMCC': 'VSFK', 'OCoCr': 'IBTV', 'OId-BDTA': 'IdTVD-V', 'OIBDTA': 'IdTVD-V', 'LIPPI': 'IFEG', 'LIPIn': 'IFEG', 'OIPI': 'IGE-OV', 'Oper-IPI': 'IGE-PersV', 'OPer-IPI': 'IGE-PersV', 'LIPI': 'IGEG', 'OAiR': 'InkHV', 'OAInc': 'InkHV', 'Oinv': 'InvV', 'OPICin': 'IPFiV', 'LDIP': 'IPRG', 'LISint': 'IQG', 'LIFI': 'IQG', 'RInfo-TFB': 'IR-PatGer', 'EIMP': 'IRSG', 'AIMP': 'IRSG', 'OEIMP': 'IRSV', 'OAIMP': 'IRSV', 'O-SI ABV': 'ISABV-V', 'O-SIAMV': 'ISABV-V', 'LSI': 'ISG', 'LSIn': 'ISG', 'O-SICAL': 'ISLK-V', 'O-SIFA': 'ISLK-V', 'OSIAgr': 'ISLV', 'OSIP-OFDF': 'IStrV-BAZG', 'OSIP-UDSC': 'IStrV-BAZG', 'OSAR': 'ISUV', 'OSIStr': 'ISUV', 'ODiv': 'IvDV', 'ODIV': 'IvDV', 'LAI': 'IVG', 'RAI': 'IVV', 'OAI': 'IVV', 'OSIAC': 'IVZV', 'O OFSPO J+S': 'J+S-V-BASPO', 'O-G+S-UFSPO': 'J+S-V-BASPO', 'LPMFJ': 'JSFVG', 'LPMFV': 'JSFVG', 'OPMFJ': 'JSFVV', 'OPMFV': 'JSFVV', 'LChP': 'JSG', 'LCP': 'JSG', 'DPMin': 'JStG', 'PPMin': 'JStPO', 'OChP': 'JSV', 'OCP': 'VKL', 'OFPC-FINMA': 'KAKV-FINMA', 'OFICol-FINMA': 'KAKV-FINMA', 'OAAcc': 'KBFHV', 'OACust': 'KBFHV', 'LENu': 'KEG', 'OENu': 'KEV', 'LEC': 'KFG', 'LPCu': 'KFG', 'OAVAuto': 'KFZV', 'OAuto': 'KFZV', 'LTBC': 'KGTG', 'OTBC': 'KGTV', 'OIBC': 'KGVV', 'OEBC': 'KGVV', 'LRCN': 'KHG', 'ORCN': 'KHV', 'LIC': 'KIG', 'LEEJ': 'KJFG', 'LPAG': 'KJFG', 'OEEJ': 'KJFV', 'OPAG': 'KJFV', 'LCC': 'KKG', 'OPCC': 'KKV', 'OICol': 'KKV', 'OPC-FINMA': 'KKV-FINMA', 'OICol-FINMA': 'KKV-FINMA', 'OClin': 'KlinV', 'OSRUm': 'KlinV', 'OClin-Dim': 'KlinV-Mep', 'OSRUm-Dmed': 'KlinV-Mep', 'OCI': 'KlV', 'OOCli': 'KlV', 'LFMG': 'KMG', 'LMB': 'KMG', 'OMG': 'KMV', 'OMB': 'KMV', 'OAOF': 'KOV', 'RUF': 'KOV', 'OCoo': 'KoVo', 'OCCRT': 'KoVo', 'OAMédcophy': 'KPAV', 'OMCF': 'KPAV', 'OCPF': 'KPFV', 'FP-TFB': 'KR-PatGer', 'SpG-TFB': 'KR-PatGer', 'OCré-FINMA': 'KreV-FINMA', 'OCre-FINMA': 'KreV-FINMA', 'OEMO': 'KRV', 'ORMT': 'KRV', 'Cst-GE': 'KV-GE', 'Cost.-GE': 'KV-GE', 'LSAMal': 'KVAG', 'LVAMal': 'KVAG', 'LAMal': 'KVG', 'OAMal': 'KVV', 'OPVA': 'LAfV', 'OPSAgr': 'LAfV', 'LRA': 'LBG', 'OAgrD': 'LDV', 'ODAgr': 'LDV', 'OLEl': 'LeV', 'LA': 'LFG', 'LNA': 'LFG', 'OSAv': 'LFV', 'ONA': 'LFV', 'OIBL': 'LGBV', 'OULiq': 'LGBV', 'OGN': 'LGeoIV', 'ODAlOUs': 'LGV', 'ODerr': 'LGV', 'OLiq': 'LiqV', 'OIDAl': 'LIV', 'OID': 'LIV', 'LDAl': 'LMG', 'LDerr': 'LMG', 'OIMLo': 'LMmV', 'OSML': 'LMmV', 'OELDAl': 'LMVV', 'OELDerr': 'LMVV', 'LBFA': 'LPG', 'LAAgr': 'LPG', 'OLRO-FINMA': 'LROV-FINMA', 'OPair': 'LRV', 'OIAt': 'LRV', 'LPPM': 'LSMG', 'OPPM': 'LSMV', 'OPB': 'LSV', 'OIF': 'LSV', 'OTrA': 'LTrV', 'CL': 'LugÜ', 'CLug': 'LugÜ', 'LAP': 'LVG', 'OMN': 'LVV', 'OMN-DDPS': 'LVV-VBS', 'LAgr': 'LwG', 'OAcCP': 'MAkkV', 'OAGio': 'MAkkV', 'OBMa': 'MaLV', 'ORMAp': 'MaLV', 'OMar-FINMA': 'MarV-FINMA', 'OMer-FINMA': 'MarV-FINMA', 'OMach': 'MaschV', 'OMacch': 'MaschV', 'OMat': 'MatV', 'OMAT-DDPS': 'MatV', 'ORM': 'MAV', 'OPart': 'MBV', 'OParC': 'MBV', 'OCMil': 'MCAV', 'OCDM': 'MCAV', 'ODqua': 'MeAV', 'OIQ': 'MeAV', 'ODqua-DFJP': 'MeAV-EJPD', 'OIQ-DFGP': 'MeAV-EJPD', 'LPMéd': 'MedBG', 'LPMed': 'MedBG', 'OPMéd': 'MedBV', 'OPMed': 'MedBV', 'ODim': 'MepV', 'ODmed': 'MepV', 'OSRM': 'MeQV', 'LMétr': 'MessG', 'LMetr': 'MessG', 'OIMes': 'MessMV', 'OStrM': 'MessMV', 'LMét': 'MetG', 'LMet': 'MetG', 'OMét': 'MetV', 'OMet': 'MetV', 'OSV': 'MFV', 'OSVM': 'MFV', 'LAAM': 'MG', 'LM': 'MG', 'OBCBA': 'MGwV', 'OURD': 'MGwV', 'LSIA': 'MIG', 'LSIM': 'MIG', 'OIMin': 'MindStV', 'OImM': 'MindStV', 'OMinTA': 'MinLV', 'Limpmin': 'MinöStG', 'LIOm': 'MinöStG', 'Oimpmin': 'MinöStV', 'OIOm': 'MinöStV', 'LUMin': 'MinVG', 'OUMin': 'MinVV', 'OCL': 'MiPV', 'OSIAr': 'MIV', 'OSIM': 'MIV', 'OCM ES': 'MiVo-HF', 'OERic-SSS': 'MiVo-HF', 'OJM': 'MJV', 'O-GM': 'MJV', 'OAMil': 'MLFV', 'OPCNP': 'MNKPV', 'LPM': 'MSchG', 'OPM': 'MSchV', 'CPM': 'MStG', 'PPM': 'MStP', 'OJPM': 'MStV', 'OGPM': 'MStV', 'O sur la monnaie': 'MünzV', 'OMon': 'MünzV', 'LAM': 'MVG', 'OAM': 'MVV', 'LTVA': 'MWSTG', 'LIVA': 'MWSTG', 'OTVA': 'MWSTV', 'OIVA': 'MWSTV', 'LFORTA': 'NAFG', 'LFOSTRA': 'NAFG', 'ONag': 'NagV', 'OANS': 'NARV', 'OPANS': 'NARV', 'LBN': 'NBG', 'LBNS': 'NBibG', 'OBNS': 'NBibV', 'LRens': 'NDG', 'LAIn': 'NDG', 'ORens': 'NDV', 'OAIn': 'NDV', 'OMBT': 'NEV', 'OPBT': 'NEV', 'OPU': 'NFSV', 'OPE': 'PAVO', 'LPN': 'NHG', 'OPN': 'NHV', 'LRNIS': 'NISSG', 'ORNI': 'NISV', 'OIBT': 'NIV', 'OPCN-OCDE': 'NKPV-OECD', 'OPCN-OCSE': 'NKPV-OECD', 'LVA': 'NSAG', 'LUSN': 'NSAG', 'OVA': 'NSAV', 'OUSN': 'NSAV', 'LRN': 'NSG', 'LSN': 'NSG', 'ORN': 'NSV', 'OSN': 'NSV', 'OARF': 'NZV', 'OARF-OFT': 'NZV-BAV', 'OARF-UFT': 'NZV-BAV', 'OHS-LP': 'OAV-SchKG', 'OAV-LEF': 'OAV-SchKG', 'LAO': 'OBG', 'LMD': 'OBG', 'OAO': 'VUB', 'OMD': 'OBV', 'OPub-FINMA': 'OffV-FINMA', 'LAVI': 'OHG', 'LAV': 'OHG', 'OAVI': 'OHV', 'CO': 'OR', 'OCRDP': 'ÖREBKV', 'OCRDPP': 'ÖREBKV', 'OdelO': 'OrFV', 'OTOr': 'OrFV', 'Org-OMP': 'Org-VöB', 'OOAPub': 'Org-VöB', 'Org ChF': 'OV-BK', 'OrgCaF': 'OV-BK', 'Org CF': 'OV-BR', 'OOrg-CF': 'OV-BR', 'Org DFAE': 'OV-EDA', 'OOrg-DFAE': 'OV-EDA', 'Org DFI': 'OV-EDI', 'OOrg-DFI': 'OV-EDI', 'Org DFF': 'OV-EFD', 'Org-DFF': 'OV-EFD', 'Org DFJP': 'OV-EJPD', 'Org-DFGP': 'OV-EJPD', 'Org LRH': 'OV-HFG', 'Org-LRUm': 'OV-HFG', 'Org DETEC': 'OV-UVEK', 'Org-DATEC': 'OV-UVEK', 'Org-DDPS': 'OV-VBS', 'OOrg-DDPS': 'OV-VBS', 'Org DEFR': 'OV-WBF', 'Org-DEFR': 'OV-WBF', 'LCBr': 'PAG', 'LParl': 'ParlG', 'OLPA': 'VABFP', 'Oparl': 'ParlVV', 'LPart': 'PartG', 'LUD': 'PartG', 'OPTP': 'PaRV', 'OPFP': 'PaRV', 'LBI': 'PatG', 'LTFB': 'PatGG', 'OParcs': 'PäV', 'OPar': 'PäV', 'OAMin': 'PAVO', 'OPTA': 'PAVV', 'LTV': 'PBG', 'OPD-EPF': 'PDV-ETH', 'OPDP-PF': 'PDV-ETH', 'LLG': 'PfG', 'LOF': 'PfG', 'OLG': 'PfV', 'OOF': 'PfV', 'OSaVé': 'PGesV', 'OSalV': 'PGesV', 'OSaVé-DEFR-DETEC': 'PGesV-WBF-UVEK', 'OSalV-DEFR-DATEC': 'PGesV-WBF-UVEK', 'ORPGAA': 'PGRELV', 'ORFGAA': 'PGRELV', 'OPha': 'PhaV', 'OFarm': 'PhaV', 'LOP': 'POG', 'LRFP': 'PrHG', 'LRDP': 'PrHG', 'LSPro': 'PrSG', 'OSPro': 'PrSV', 'ORRTP': 'PRTR-V', 'OPRTR': 'PRTR-V', 'OEPI': 'PSAV', 'ODPI': 'PSAV', 'OPPh': 'PSMV', 'OPF': 'VSB', 'OPsy': 'PsyBV', 'OPPsi': 'PsyBV', 'LPsy': 'PsyG', 'LPPsi': 'PsyG', 'LSPr': 'PüG', 'OPers-METAS': 'PV-METAS', 'OPersTF': 'PVBger', 'OPers-PDHH': 'PVFMH', 'OPers-PRA': 'PVFMH', 'OPers-PDHH-DDPS': 'PVFMH-VBS', 'OPers-PRA-DDPS': 'PVFMH-VBS', 'OPersT': 'PVGer', 'OPers-EPF': 'PVO-ETH', 'OPers PF': 'PVO-ETH', 'OPers-ServAS': 'PVO-TVS', 'OPers-SAT': 'PVO-TVS', 'OPers-PPOE': 'PVSPA', 'OPers-POE': 'PVSPA', 'OPers-PPOE-DDPS': 'PVSPA-VBS', 'OPers-POE-DDPS': 'PVSPA-VBS', 'OIS': 'QStV', 'OIFo': 'QStV', 'OQuaDu': 'QuNaV', 'OQuSo': 'QuNaV', 'R-COPA': 'R-UEK', 'LSR': 'RAG', 'OSRev': 'RAV', 'OEPC-FINMA': 'RelV-FINMA', 'OAPC-FINMA': 'RelV-FINMA', 'RCETF': 'ReRBGer', 'ORe-DFI': 'ResV-EDI', 'ORAMal-DFI': 'ResV-EDI', 'LHR': 'RHG', 'LArRa': 'RHG', 'OHR': 'RHV', 'OArRa': 'RHV', 'OSITC': 'RLSV', 'OITC': 'RLV', 'OrX': 'RöV', 'LAT': 'RPG', 'LPT': 'RPG', 'OAT': 'RPV', 'OPT': 'RPV', 'LRTV': 'RTVG', 'ORTV': 'RTVV', 'OR-AVS': 'RV-AHV', 'LOGA': 'RVOG', 'OLOGA': 'RVOV', 'LASEI': 'SAFIG', 'LASPI': 'SAFIG', 'OPTM': 'SAMV', 'OPLM': 'SAMV', 'OAGa': 'SaV', 'OASal': 'SaV', 'LCFF': 'SBBG', 'LFFS': 'SBBG', 'OMAS': 'SBMV', 'OMSC': 'SBMV', 'LP': 'SchKG', 'LEF': 'SchKG', 'LICa': 'SebG', 'LIFT': 'SebG', 'OICa': 'SebV', 'OIFT': 'SebV', 'OFDG': 'SEFV', 'OFDS': 'SEFV', 'OCâbles': 'SeilV', 'OFuni': 'SeilV', 'OASRE': 'SERV-V', 'OARE': 'SERV-V', 'LASRE': 'SERVG', 'LARE': 'SERVG', 'OPAAb': 'SGV', 'OPeM': 'SGV', 'LEIS': 'SIaG', 'LISDC': 'SIRG', 'CRUGBIN': 'BASEVKGNMD', 'ACSRUGBIN': 'BASEVKGNMD', 'ORIn': 'SnAV', 'ORim': 'SnAV', 'OMJ-DFJP': 'SPBV-EJPD', 'OCG-DFGP': 'SPBV-EJPD', 'OSLing': 'SpDV', 'LLC': 'SpG', 'LLing': 'SpG', 'LESp': 'SpoFöG', 'LPSpo': 'SpoFöG', 'OESp': 'SpoFöV', 'OPSpo': 'SpoFöV', 'LExpl': 'SprstG', 'LEspl': 'SprstG', 'OExpl': 'SprstV', 'OEspl': 'SprstV', 'OLang': 'SpV', 'OLing': 'SpV', 'LVP': 'SRVG', 'LPV': 'SRVG', 'LESE': 'SSchG', 'LSSE': 'SSchG', 'OESE': 'SSchV', 'OSSE': 'SSchV', 'OESE-DFI': 'SSchV-EDI', 'OSSE-DFI': 'SSchV-EDI', 'LNM': 'SSG', 'OSR': 'SSV', 'OSStr': 'SSV', 'LECF': 'StADG', 'LOA': 'StAG', 'LImA': 'StAG', 'LAAF': 'StAhiG', 'OAAF': 'StAhiV', 'OSOA': 'StAV', 'OImA': 'StAV', 'LOAP': 'StBOG', 'OASF': 'STEBV', 'LRCS': 'StFG', 'LCel': 'StFG', 'OPAM': 'StFV', 'OPIR': 'StFV', 'CP': 'StGB', 'LHID': 'StHG', 'LAID': 'StHG', 'OIMRI': 'StMmV', 'OSMRI': 'StMmV', 'CPP': 'StPO', 'LCJ': 'StReG', 'LCaGi': 'StReG', 'OCJ': 'StReV', 'OCaGi': 'StReV', 'LApEl': 'StromVG', 'LAEl': 'StromVG', 'OApEl': 'StromVV', 'OAEl': 'StromVV', 'LRaP': 'StSG', 'ORaP': 'StSV', 'LEnTR': 'STUG', 'LPTS': 'STUG', 'OEnTR': 'STUV', 'OTStr': 'STUV', 'LTRA': 'STVG', 'LTS': 'STVG', 'LSu': 'SuG', 'LRPL': 'SVAG', 'LTTP': 'SVAG', 'ORPL': 'SVAV', 'OTTP': 'SVAV', 'LCR': 'SVG', 'LCStr': 'SVG', 'OS LCart': 'SVKG', 'LPTab': 'TabPG', 'OPTab': 'TabPV', 'OETV 1': 'TAFV 1', 'OETV 2': 'TAFV 2', 'OETV 3': 'TAFV 3', 'OMédv': 'TAMV', 'OMvet': 'TAMV', 'OPBD': 'TBDV', 'OPPD': 'TBDV', 'LVPC': 'TEVG', 'LRVC': 'TEVG', 'OTRF': 'TGBV', 'OSSAn': 'TGDV', 'LETC': 'THG', 'LOTC': 'THG', 'OIMTh': 'TMmV', 'OSMT': 'TMmV', 'LTo': 'ToG', 'OTo': 'ToV', 'OFPT': 'TPFV', 'LTro': 'TrG', 'LIF': 'TrG', 'OFPAn': 'TSchAV', 'LPA': 'TSchG', 'LPAn': 'TSchG', 'OPAn': 'TSchV', 'LFE': 'TSG', 'LTab': 'TStG', 'LImT': 'TStG', 'OITab': 'TStV', 'OImT': 'TStV', 'LET': 'TUG', 'LATC': 'TUG', 'OServAS': 'TVSV', 'OSAT': 'TVSV', 'OPPPS': 'TwwV', 'OPPS': 'VMD', 'LACRE': 'UEG', 'LSgrI': 'UEG', 'OOPA': 'UEV', 'O-COPA': 'UEV', 'Règlement sur la pêche dans le lac Inférieur': 'Unterseefischereiordnung', 'Ordinamento della pesca nel Lago Inferiore': 'Unterseefischereiordnung', 'LTSM': 'UGüTG', 'LTMS': 'UGüTG', 'LIDE': 'UIDG', 'LIDI': 'UIDG', 'OIDE': 'UIDV', 'OIDI': 'UIDV', 'LPtra': 'ÜLG', 'LPTD': 'ÜLG', 'OPtra': 'ÜLV', 'OPTD': 'ÜLV', 'OUMR': 'UraM', 'MMRa': 'UraM', 'LDA': 'URG', 'ODAu': 'URV', 'LPE': 'USG', 'LPAmb': 'USG', 'LAA': 'UVG', 'LAINF': 'UVG', 'OEIE': 'UVPV', 'OEIA': 'UVPV', 'OLAA': 'UVV', 'OAINF': 'UVV', 'LCD': 'UWG', 'LCSl': 'UWG', 'OPersMil': 'V Mil Pers', 'OPers mil': 'V Mil Pers', 'OSEtr': 'V-ASG', 'OSEst': 'V-ASG', 'O-LERI': 'V-FIFG', 'O-LPRI': 'V-FIFG', 'O-LERI-DEFR': 'V-FIFG-WBF', 'O-LPRI-DEFR': 'V-FIFG-WBF', 'OLEH': 'V-GSG', 'OSOsp': 'V-GSG', 'O-LEHE': 'V-HFKG', 'O-LPSU': 'V-HFKG', 'O-STAC': 'V-LTDB', 'O-SIEs': 'V-NDA', 'O-LRNIS': 'V-NISSG', 'O-CNC-FPr': 'V-NQR-BB', 'O QNQ FP': 'V-NQR-BB', 'OCommcon-EPF': 'V-Schliko-ETH', 'O CommConc PF': 'V-Schliko-ETH', 'O-AQIS': 'V-SQWI', 'O-GQIS': 'V-SQWI', 'O-CP-CPM': 'V-StGB-MSt', 'OCP-CPM': 'V-StGB-MSt', 'OPNA': 'VABFP', 'OCDoc': 'VABK', 'RCDoc': 'VABK', 'OETHand': 'VAböV', 'ORTDis': 'VAböV', 'OISofCA': 'VABUA', 'OFSPE': 'VABUA', 'OCFH': 'VAEW', 'OIFI': 'VAEW', 'LSA': 'VAG', 'OMHSI': 'VAI', 'OMFSI': 'VAI', 'OIFC': 'VAK', 'OCFQE': 'VAK', 'OMéd': 'VAM', 'OM': 'VAM', 'OIMEC': 'VAMF', 'OMGC': 'VAMF', 'OMSVM': 'VAmFD', 'OIGE': 'VAMV', 'OSGS': 'VAMV', 'Odac': 'VAN', 'OEAC': 'VAN', 'OSRens': 'VAND', 'OVAIn': 'VAND', 'OLPS': 'VAPF', 'OQPN': 'VAPK', 'OEPIN': 'VAPK', 'OTAS': 'VASA', 'OTaRSi': 'VASA', 'OMBat': 'VASm', 'ONCR': 'VASR', 'OITEP': 'VATPE', 'OITIP': 'VATPE', 'OAHST': 'VATT', 'OATFS': 'VATT', 'OAAFM': 'VATV', 'OASAM': 'VATV', 'OAAFM-DDPS': 'VATV-VBS', 'OASAM-DDPS': 'VATV-VBS', 'ODPO': 'VAU', 'OAPO': 'VAU', 'OInstr pré': 'VAusb', 'OISP': 'VAusb', 'OInstr prém DDPS': 'VAusb-VBS', 'OISP-DDPS': 'VAusb-VBS', 'OMO': 'VAV', 'OMU': 'VAV', 'OMO-DDPS': 'VAV-VBS', 'OMU-DDPS': 'VAV-VBS', 'OLDI': 'VAwG', 'ODI': 'VID', 'OASMéd': 'VAZV', 'OOSM': 'VAZV', 'ODFR': 'VBB', 'Osol': 'VBBo', 'O suolo': 'VBBo', 'OLAr': 'VBGA', 'OLFP': 'VBGF', 'OTrans': 'VBGÖ', 'OTras': 'VBGÖ', 'OIFP': 'VBLN', 'OTUIC': 'VBNIB', 'ODO': 'VBO', 'OOC-SCPT': 'VBO-ÜPF', 'OThand': 'VböV', 'OTDis': 'VböV', 'OPBio': 'VBP', 'OIOP': 'VBPO', 'OOCOP': 'VBPO', 'O-OPers': 'VBPV', 'O-OPers-DFAE': 'VBPV-EDA', 'ORE I': 'VBR I', 'ONE I': 'VBR I', 'ORCSN': 'VBRK', 'ORTN': 'VBRK', 'OEMFP': 'VBSTB', 'OSMFP': 'VBSTB', 'OPSEnt': 'VBSV', 'OPSAz': 'VBSV', 'OAdma': 'VBVA', 'OAE-AF': 'VBVA', 'OGPCT': 'VBVV', 'OABCT': 'VBVV', 'OESN': 'VBWK', 'OCGIN': 'VBWK', 'OCITES': 'VCITES', 'O-CITES': 'VCITES', 'OME-SCPT': 'VD-ÜPF', 'OE-SCPT': 'VD-ÜPF', 'OODA': 'VDA', 'OODE': 'VDA', 'OTPSP': 'VDPS', 'ODPSP': 'VDPS', 'OEDRP-DFI': 'VDPV-EDI', 'OSDRP-DFI': 'VDPV-EDI', 'OCPD': 'VDSZ', 'OTNI': 'VDTI', 'OTDI': 'VDTI', 'OACA': 'VDZV', 'ODCA': 'VDZV', 'OLQE': 'VEAF', 'OLAE': 'VEAF', 'OIELFP': 'VEAGOG', 'OIEVFF': 'VEAGOG', 'OEFMP': 'VEBMP', 'OEE-VVT': 'VEE-PLS', 'OEE-AAT': 'VEE-PLS', 'ODF': 'VEJ', 'OBAF': 'VEJ', 'OGE': 'VEKF', 'OCGE': 'VEKF', 'OEmiA': 'VEL', 'OVotE': 'VEleS', 'OVE': 'VEleS', 'OMETr': 'VEMZ', 'OSTAC': 'VEMZ', 'OESS': 'VES', 'OIIS': 'VES', 'OCREPF': 'VETHBK', 'OCRPF': 'VETHBK', 'OCEl-PA': 'VeÜ-VwV', 'OCE-PA': 'VeÜ-VwV', 'OCEl-PCPP': 'VeÜ-ZSSV', 'OCE-PCPE': 'VeÜ-ZSSV', 'OEV': 'VEV', 'OMoD': 'VeVA', 'OTRif': 'VeVA', 'O E-VERA': 'VEVERA', 'Ordinanza E-VERA': 'VEVERA', 'OIMA': 'VLIb', 'OIMeA': 'VFAI', 'OAFA': 'VFAL', 'OOIT': 'VFAV', 'OPer-Fu': 'VFB-B', 'OLAFum': 'VFB-B', 'OPer-D': 'VFB-DB', 'OADAP': 'VFB-DB', 'OPer-H': 'VFB-G', 'OAS-O': 'VFB-G', 'OPer-B': 'VFB-H', 'OASL': 'VFB-H', 'OPer-Fl': 'VFB-K', 'OASPR': 'VFB-K', 'OPer-AH': 'VFB-LG', 'OASAOG': 'VFB-LG', 'OPer-P': 'VFB-S', 'OALPar': 'VFB-S', 'OPer-S': 'VFB-SB', 'OASSP': 'VFB-SB', 'OPer-Fo': 'VFB-W', 'OASEF': 'VFB-W', 'OVCC': 'VFBF', 'OMA': 'VFD', 'OILAE': 'VFlaW', 'OLDDA': 'VFlaW', 'OLCP': 'VFP', 'Oform': 'VFRR', 'Rform': 'VFRR', 'OSNA': 'VFSD', 'OSA': 'VFSD', 'OAF': 'VFV', 'LRCF': 'VG', 'LResp': 'VG', 'OFCoop': 'VGeK', 'RFCoop': 'VGeK', 'LTAF': 'VGG', 'FITAF': 'VGKE', 'TS-TAF': 'VGKE', 'RTAF': 'VGR', 'OJAr': 'VGS', 'OGD': 'VGS', 'OSPEX': 'VGSEB', 'OASAE': 'VGSEB', 'OEB': 'VGV', 'OIB': 'VGV', 'ODAlGM': 'VGVL', 'ODerrGM': 'VGVL', 'ORegBL': 'VGWR', 'OREA': 'VREG', 'OGOC': 'VHBT', 'OGOCC': 'VHBT', 'OCont': 'VHK', 'OHyPL': 'VHyMP', 'OIgPL': 'VHyMP', 'OHyPPr': 'VHyPrP', 'OIPPrim': 'VHyPrP', 'OHyAb': 'VHyS', 'OIgM': 'VHyS', 'ODIn': 'VID', 'OSIA': 'VIL', 'OILC': 'VILB', 'OSIC': 'VIM', 'OICoM': 'VIM', 'OCMI': 'VIMK', 'OIE': 'VIntA', 'OIntS': 'VIntA', 'OPPEtr': 'VIPaV', 'OIPPE': 'VIPaV', 'OSIS-SRC': 'VIS-NDB', 'OSIME-SIC': 'VIS-NDB', 'OISOS': 'VISOS', 'OVIS': 'VISV', 'OITPTh': 'VITH', 'OITAT': 'VITH', 'OIVS': 'VIVS', 'OCISF': 'ViZG', 'OCISC': 'ViZG', 'OCMIF': 'VIZMB', 'OJAR-FSTD': 'VJAR-FSTD', 'OACata': 'VKA', 'OLCC': 'VKKG', 'OCCEA': 'VKKL', 'OCoC': 'VKKL', 'OSPBC': 'VKKP', 'OSBC': 'VKKP', 'OCPre': 'VKL', 'OCSN': 'VKNS', 'OCos': 'VKos', 'OCTSE': 'VKOVA', 'OPFr': 'VKP', 'OCPPME': 'VKP-KMU', 'OCPPMI': 'VKP-KMU', 'OSSC': 'VKSD', 'Ocach': 'VKSWk', 'OCRCI': 'VKSWk', 'OFA-FINMA': 'VKV-FINMA', 'OCTrI': 'VKVöV', 'OCTrIn': 'VKVöV', 'OMDA': 'VKZ', 'OCA': 'VVK', 'OBNP': 'VLBE', 'ODPPE': 'VLBE', 'OBCF': 'VLE', 'ORFF': 'VLE', 'ORAgr': 'VLF', 'LCo': 'VlG', 'OECA': 'VLHb', 'OICA': 'VLHb', 'OOMA': 'VLIb', 'OPEA': 'VLIp', 'OCDA': 'VLKA', 'OCDAE': 'VLKA', 'ONAE': 'VLL', 'ODNA': 'VLL', 'ODAlOV': 'VLpH', 'ODOV': 'VLpH', 'ODAlAn': 'VLtH', 'ODOA': 'VLtH', 'OCo': 'VlV', 'OEMTP': 'VMAP', 'OSMLP': 'VMAP', 'OAMAS': 'VMBM', 'OAMM': 'VMBM', 'ODPS': 'VMD', 'OMi': 'VMDP', 'OOPSM': 'VMDP', 'OACAMIL': 'VMILAK', 'OACMIL': 'VMILAK', 'OAMC': 'VmKI', 'OMob': 'VMob', 'OSM': 'VMS', 'ONM': 'VMSch', 'OCM': 'VMSV', 'OCSM': 'VMSV', 'OBLF': 'VMWG', 'OLAL': 'VMWG', 'OOPT': 'VNEF', 'OORT': 'VNEF', 'OCNE': 'VNEK', 'OCAl': 'VNem', 'OIAl': 'VNem', 'OUS': 'VNF', 'OAPub': 'VöB', 'OCOV': 'VOCV', 'OSO': 'VOD', 'OOBE': 'VOEW', 'OOSE': 'VOEW', 'OOSG': 'VOGW', 'OCoR': 'VORA', 'OCoR-DFI': 'VORA-EDI', 'OTN': 'VOSA', 'OPoA': 'VPA', 'OPPE': 'VPA', 'ORCPP': 'VPABP', 'OPPCPers': 'VPABP', 'OSAss': 'VPAV', 'RPAss': 'VPAV', 'OTV': 'VPB', 'OPIE': 'VPeA', 'OPAR': 'VPEV', 'ORPAR': 'VPEV', 'OAPA': 'VPGA', 'ORInt': 'VPiB', 'OMP-OFEV': 'VpM-BAFU', 'OMF-UFAM': 'VpM-BAFU', 'OMP-OFAG': 'VpM-BLW', 'OMF-UFAG': 'VpM-BLW', 'OOP EPF': 'VPO ETH', 'OOP-PF': 'VPO ETH', 'OOPC': 'VPOB', 'OFipo': 'VPofi', 'OFiPo': 'VPofi', 'OLOP': 'VPOG', 'OOP': 'VPOG', 'ODP': 'VPR', 'OMAP': 'VPRG', 'ODFCLSI': 'VPRG', 'OPOVA': 'VPRH', 'OAOVA': 'VPRH', 'OPPr': 'VPrP', 'OPPrim': 'VPrP', 'OPSP': 'VPS', 'OCSP': 'VPSP', 'OEnB': 'VPV', 'RPB': 'VPV', 'OPAPIF': 'VPVE', 'ORPM': 'VPVK', 'ORPMUE': 'VPVKEU', 'RP-EPF 1': 'VR-ETH 1', 'RP-PF 1': 'VR-ETH 1', 'RP-EPF 2': 'VR-ETH 2', 'RP-PF 2': 'VR-ETH 2', 'OCBD': 'VRA', 'OCQO': 'VRA', 'RPEC': 'VRAB', 'RPIC': 'VRAB', 'ORSAE': 'VREG', 'OSCR': 'VRKD', 'ORésDAlan': 'VRLtH', 'ORDOA': 'VRLtH', 'OPR': 'VRP', 'ORCPL': 'VRSL', 'OReq': 'VRSL', 'ORA': 'VRV-L', 'ONCA': 'VRV-L', 'OPCF': 'VSB', 'OEMCN': 'VSBN', 'OSMCN': 'VSBN', 'OAbCV': 'VSFK', 'ODCS': 'VSFS', 'ODSRS': 'VSFS', 'OOCCR-OFROU': 'VSKV-ASTRA', 'OOCCS-USTR': 'VSKV-ASTRA', 'OMSA': 'VSL', 'OSMP': 'VSMS', 'OMSM': 'VSMS', 'ODiTr': 'VSoTr', 'ODiT': 'VSoTr', 'OPPBE': 'VSPA', 'OPBE': 'VSPA', 'OPESp': 'VSpoFöP', 'OPPSpo': 'VSpoFöP', 'OPPB': 'VSPS', 'ORS': 'VSR', 'ORSA': 'VSRL', 'OSJo': 'VSS', 'OSG': 'VSS', 'OOST': 'VST', 'OOSI': 'VST', 'ORCS': 'VStFG', 'ORCel': 'VStFG', 'LIA': 'VStG', 'LIP': 'VStG', 'DPA': 'VStrR', 'OIA': 'VStV', 'OIPrev': 'VStV', 'OSAA': 'VSUV', 'OSAI': 'VSUV', 'OFDPP': 'VSVB', 'OFDP': 'VSVB', 'OEIT': 'VSZV', 'OIET': 'VSZV', 'OCVM': 'VTE', 'OVF': 'VTE', 'OAP': 'VTM', 'OAAP': 'VTM', 'OSPA': 'VTNP', 'OSOAn': 'VTNP', 'OETV': 'VTS', 'OPAnAb': 'VTSchS', 'OPAnMac': 'VTSchS', 'OPAT': 'VWS', 'OPrTec': 'VtVtH', 'OOr': 'VUB', 'OOr-DEFR': 'VUB-WBF', 'OAO-DEFR': 'VUB-WBF', 'OFSPers': 'VUFB', 'OCPCP': 'VUKBK', 'OCIPP': 'VUKBK', 'OACM': 'VUM', 'OAAM': 'VUM', 'OSCPT': 'VÜPF', 'OPA': 'VUV', 'OPI': 'VUV', 'OVid-TP': 'VüV-ÖV', 'OVsor-TP': 'VüV-ÖV', 'OROPD': 'VUZPE', 'OROPS': 'VUZPE', 'OAA': 'VVA', 'OAE': 'VVA', 'OPC': 'VVAG', 'ODiC': 'VVAG', 'OOLDI': 'VVAwG', 'O-ODI-DFAE': 'VVAwG', 'OISec': 'VVE', 'OFIM': 'VVE', 'OLED': 'VVEA', 'OPSR': 'VVEA', 'LCA': 'VVG', 'OTeA': 'VVK', 'OCA-DFI': 'VVK-EDI', 'OTeA-DFI': 'VVK-EDI', 'OMAH': 'VVMH', 'OMPAH': 'VVMH', 'OOUS': 'VVNF', 'OUUS': 'VVNF', 'OST-SCPT': 'VVS-ÜPF', 'OPSE': 'VVSG', 'OPreS': 'VVSG', 'OERE': 'VVWAL', 'OEAE': 'VVWAL', 'LALM': 'VWBG', 'LMAM': 'VWBG', 'OLCAP': 'VWEG', 'OFSI': 'VWEV', 'OCMSD': 'VWEV', 'OSS': 'VWL', 'OAEP': 'VWLV', 'OPATE': 'VWS', 'PA': 'VwVG', 'OASA': 'VZAE', 'LGEL': 'VZEG', 'LPIL': 'VZEG', 'OSCSE': 'VZertES', 'OFiEle': 'VZertES', 'ORFI': 'VZG', 'RFF': 'VZG', 'ONCAF': 'VZSchB', 'OAC': 'VZV', 'OASM': 'VZVM', 'OAVM': 'VZVM', 'LFo': 'WaG', 'OFo': 'WaV', 'LFCo': 'WeBiG', 'OFCo': 'WeBiV', 'OEPL': 'WEFV', 'OPPA': 'WEFV', 'LCAP': 'WEG', 'LArm': 'WG', 'LAMO': 'WHG', 'LTEO': 'WPEG', 'OTEO': 'WPEV', 'OIRH': 'WResV', 'OREI': 'WResV', 'LFH': 'WRG', 'LUFI': 'WRG', 'OFH': 'WRV', 'OUFI': 'WRV', 'LPAP': 'WSchG', 'LPSt': 'WSchG', 'OPAP': 'WSchV', 'OPSt': 'WSchV', 'OMT': 'WTO', 'OArm': 'WV', 'LUMMP': 'WZG', 'LUMP': 'WZG', 'RDE': 'WZV', 'ODA': 'WZV', 'OROEM': 'WZVV', 'ORUAM': 'WZVV', 'LUsC': 'ZAG', 'LCoe': 'ZAG', 'LFisE': 'ZBstG', 'LFR': 'ZBstG', 'LSC': 'ZDG', 'ODSC': 'ZDUeV', 'OSCi': 'ZDV', 'OSCi-DEFR': 'ZDV-WBF', 'LDIF': 'ZEBG', 'LSIF': 'ZEBG', 'LOC': 'ZentG', 'LUC': 'ZentG', 'SCSE': 'ZertES', 'FiEle': 'ZertES', 'Ltém': 'ZeugSG', 'LPTes': 'ZeugSG', 'OTém': 'ZeugSV', 'OPTes': 'ZeugSV', 'OADou': 'ZEV', 'OADo': 'ZEV', 'LD': 'ZG', 'CC': 'ZGB', 'LCPI': 'ZISG', 'OCMétr': 'ZMessV', 'OCMetr': 'ZMessV', 'CPC': 'ZPO', 'CCoop-ESF': 'ZSAV-BiZ', 'CColl-SFS': 'ZSAV-BiZ', 'CCoop-HE': 'ZSAV-HS', 'ConSU': 'ZSAV-HS', 'OAASF': 'ZSTEBV', 'OEEC': 'ZStGV', 'OESC': 'ZStGV', 'OEC': 'ZStV', 'OSC': 'ZStV', 'OPCi': 'ZSV', 'LTaD': 'ZTG', 'LTD': 'ZTG', 'LAS': 'ZUG', 'OComp-OSPro': 'ZustV-PrSV', 'OAdd': 'ZuV', 'OD': 'ZV', 'OD-OFDF': 'ZV-BAZG', 'OD-UDSC': 'ZV-BAZG', 'OD-DFF': 'ZV-EFD', 'OA-DFJP': 'ZV-EJPD', 'OA-DFGP': 'ZV-EJPD', 'LRS': 'ZWG', 'LASec': 'ZWG', 'ORSec': 'ZWV', 'OASec': 'ZWV'}
print(f'multilang entries: {len(MULTILANG_ABBR)}')

In [ ]:
import re, csv, json, pickle, time, gc, sys
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

# Cell 3 — normalization / corpus index
ART_PATTERN = re.compile(
    r'^Art\.\s*(?P<art>\d+[a-z]?(?:bis|ter|quater)?)'
    r'(?:\s*Abs\.\s*(?P<abs>\d+[a-z]?(?:bis|ter)?))?'
    r'(?:\s*lit\.\s*(?P<lit>[a-zA-Z]+))?'
    r'(?:\s*Ziff\.\s*(?P<ziff>\d+))?'
    r'\s+(?P<code>[A-Za-zÄÖÜäöüß0-9\.\-]+)$'
)
def parse_article(c):
    m = ART_PATTERN.match(c.strip()); return m.groupdict() if m else None
def normalize_whitespace(c):
    s = re.sub(r'\s+', ' ', c.strip())
    s = re.sub(r'\bArt\.(?=\d)', 'Art. ', s)
    s = re.sub(r'\bAbs\.(?=\d)', 'Abs. ', s)
    s = re.sub(r'\blit\.(?=[a-zA-Z])', 'lit. ', s)
    s = re.sub(r'\bZiff\.(?=\d)', 'Ziff. ', s)
    return s
def expand_multilang(c):
    out = [c]; m = parse_article(c)
    if m and m['code'] in MULTILANG_ABBR:
        out.append(c[:c.rfind(m['code'])] + MULTILANG_ABBR[m['code']])
    return out

print('Loading laws_de.csv ...')
laws_df = pd.read_csv(DATA / 'laws_de.csv')
laws_cits = set(laws_df['citation'].astype(str))
print(f'  laws rows: {len(laws_df):,}')

print('Scanning court_considerations.csv for citations (no dense indexing in Phase 2) ...')
court_cits = set()
with open(DATA / 'court_considerations.csv', newline='', encoding='utf-8') as f:
    r = csv.reader(f); next(r, None)
    for row in r:
        if row: court_cits.add(row[0])
print(f'  court citations seen: {len(court_cits):,}')

corpus_cits = laws_cits | court_cits
parent_to_children = defaultdict(list)
for c in laws_cits:
    m = ART_PATTERN.match(c)
    if not m: continue
    p = m.groupdict()
    parent = f"Art. {p['art']} {p['code']}"
    if c != parent: parent_to_children[parent].append(c)
print(f'  combined corpus: {len(corpus_cits):,}, parents: {len(parent_to_children):,}')

def resolve_against_corpus(cit):
    c = normalize_whitespace(cit)
    if c in corpus_cits: return [c], 'exact'
    for v in expand_multilang(c):
        if v != c:
            if v in corpus_cits: return [v], 'multilang_abbr'
            if v in parent_to_children: return list(parent_to_children[v]), 'multilang_parent'
    if c in parent_to_children:
        return list(parent_to_children[c]), 'parent_to_children'
    p = parse_article(c)
    if p:
        parent = f"Art. {p['art']} {p['code']}"
        if parent != c and parent in corpus_cits:
            return [parent], 'child_to_parent'
        for v in expand_multilang(parent):
            if v != c:
                if v in corpus_cits: return [v], 'parent_multilang'
                if v in parent_to_children: return list(parent_to_children[v]), 'parent_multilang_children'
    return [], 'no_match'

def granularity_filter(cits):
    s = set(cits); out = []
    for c in cits:
        ch = parent_to_children.get(c)
        if ch and any(x in s for x in ch): continue
        out.append(c)
    return out
def dedup(cits):
    seen = set(); out = []
    for c in cits:
        if c in seen: continue
        seen.add(c); out.append(c)
    return out
def split_cits(s):
    if not isinstance(s, str) or not s.strip(): return []
    return [c.strip() for c in s.split(';') if c.strip()]

In [ ]:
import re, csv, json, pickle, time, gc, sys
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

# Cell 4 — BM25 index over laws_de.text (cached)
import time
from rank_bm25 import BM25Okapi

TOKEN_RE = re.compile(r'[A-Za-zÄÖÜäöüß0-9]+')
def tokenize(text):
    if not isinstance(text, str): return []
    return [t.lower() for t in TOKEN_RE.findall(text)]

bm25_path = CACHE / 'bm25.pkl'
laws_citations = laws_df['citation'].astype(str).tolist()
laws_texts = laws_df['text'].fillna('').astype(str).tolist()

if bm25_path.exists():
    print('Loading cached BM25 ...')
    with open(bm25_path, 'rb') as f: bm25 = pickle.load(f)
else:
    print('Tokenizing & building BM25 (one-time, ~1 min) ...')
    t0 = time.time()
    tokenized = [tokenize(t) for t in laws_texts]
    bm25 = BM25Okapi(tokenized)
    with open(bm25_path, 'wb') as f: pickle.dump(bm25, f)
    print(f'  done in {time.time()-t0:.1f}s')
print('BM25 ready')

In [ ]:
import re, csv, json, pickle, time, gc, sys
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

# Cell 5 — BGE-M3 dense index over laws_de.text (cached, GPU recommended)
import time
import gc
import torch
import faiss
from sentence_transformers import SentenceTransformer

# Clear any leftover GPU memory from prior failed run
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

BGE_M3_PATHS = [
    '/kaggle/input/bge-m3/bge-m3',
    '/kaggle/input/bge-m3',
    '/kaggle/input/baai-bge-m3',
    'BAAI/bge-m3',
]

def load_bge_m3():
    for p in BGE_M3_PATHS:
        try:
            print(f'  trying {p} ...')
            model = SentenceTransformer(p)
            print(f'  loaded from {p}')
            return model
        except Exception as e:
            print(f'    failed: {type(e).__name__}: {e}')
    raise RuntimeError('Could not load BGE-M3. Attach Kaggle Dataset or enable Internet for one run.')

faiss_path = CACHE / 'laws.faiss'
if faiss_path.exists():
    print('Loading cached FAISS ...')
    index = faiss.read_index(str(faiss_path))
    encoder = load_bge_m3()
    encoder.max_seq_length = 512
    if torch.cuda.is_available():
        encoder = encoder.to('cuda')
else:
    encoder = load_bge_m3()
    encoder.max_seq_length = 512                  # limit attention size
    if torch.cuda.is_available():
        encoder = encoder.to('cuda')
        try:
            encoder.half()
        except Exception:
            pass
    print('Embedding laws_de.text (~175K rows, GPU ~10 min) ...')
    t0 = time.time()
    emb = encoder.encode(
        laws_texts,
        batch_size=16,                            # smaller batch to avoid OOM
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).astype('float32')
    print(f'  embedding done in {time.time()-t0:.1f}s, shape={emb.shape}')
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    faiss.write_index(index, str(faiss_path))
    del emb
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print(f'FAISS index ready: ntotal={index.ntotal}, d={index.d}')

In [ ]:
import re, csv, json, pickle, time, gc, sys
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

# Cell 6 — Hybrid search: BM25(citation-context alias expansion) + Dense → RRF

# Multilang alias expansion (FR/IT → DE) only triggered when alias appears as the
# LAW CODE token of an Art./Article/al. citation pattern. Plain English words like
# 'LLC' or 'CO' won't false-trigger because they aren't preceded by 'Art. N '.

_CITATION_ALIAS_RE = re.compile(
    r'\b(?:Art(?:icle)?\.?|al\.?)\s*\d+[a-z]?'
    r'(?:\s*(?:Abs\.|al\.|para\.)\s*\d+)?'
    r'(?:\s*(?:lit\.|let\.)\s*[a-z])?'
    r'\s+([A-Z][A-Za-zÄÖÜäöüß0-9\.\-]+)'
)

def expand_query_de_aliases(query):
    """For each Art./al. citation in the query whose code is an FR/IT alias,
    append the DE-primary code as an extra token. Original query preserved."""
    extras = set()
    for m in _CITATION_ALIAS_RE.finditer(query):
        code = m.group(1)
        de = MULTILANG_ABBR.get(code)
        if de and de != code:
            extras.add(de)
    if not extras: return query
    return query + ' ' + ' '.join(extras)

def encode_query(text):
    return encoder.encode([text], normalize_embeddings=True, convert_to_numpy=True).astype('float32')

def rrf(rankings, weights=None, k=60):
    if weights is None: weights = [1.0] * len(rankings)
    scores = {}
    for rank_list, w in zip(rankings, weights):
        for r, cit in enumerate(rank_list):
            scores[cit] = scores.get(cit, 0.0) + w / (k + r + 1)
    return scores

def search_laws(query, top_k=200, bm25_n=500, dense_n=500):
    bm25_query = expand_query_de_aliases(query)
    bm25_scores = bm25.get_scores(tokenize(bm25_query))
    bm25_top = np.argsort(bm25_scores)[::-1][:bm25_n]
    bm25_ranking = [laws_citations[i] for i in bm25_top]
    q_emb = encode_query(query)
    _, I = index.search(q_emb, dense_n)
    dense_ranking = [laws_citations[i] for i in I[0] if i >= 0]
    fused = rrf([dense_ranking, bm25_ranking], weights=[0.6, 0.4])
    ranked = sorted(fused.items(), key=lambda kv: -kv[1])
    return ranked[:top_k]

In [ ]:
import re, csv, json, pickle, time, gc, sys
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

# Cell 7 — Pipeline: seed + default + retrieved → top-N
UNIVERSAL_DEFAULTS = ['Art. 100 Abs. 1 BGG']
TOP_N_EMIT = 20

RE_ART_Q  = re.compile(r'\bArt(?:icle)?\.?\s*(\d+[a-z]?(?:bis|ter|quater)?)'
                       r'(?:\s*(?:Abs\.|al\.|para\.)\s*(\d+))?'
                       r'(?:\s*(?:lit\.|let\.)\s*([a-z]))?'
                       r'\s+([A-Z][A-Za-zÄÖÜäöüß0-9\.\-]+)')
RE_BGE_Q  = re.compile(r'\bBGE\s+(\d+)\s+([IVX]+)\s+(\d+)(?:\s+E\.?\s*([\d\.]+))?')
RE_BGER_Q = re.compile(r'\b(\d+[A-Za-z]_\d+/\d+)(?:\s+E\.?\s*([\d\.]+))?')

def extract_seeds(query):
    seeds = []
    for m in RE_ART_Q.finditer(query):
        art, abs_, lit, code = m.groups()
        candidates = []
        if abs_ and lit:
            candidates.append(f'Art. {art} Abs. {abs_} lit. {lit} {code}')
            candidates.append(f'Art. {art} Abs. {abs_} {code}')
        elif abs_:
            candidates.append(f'Art. {art} Abs. {abs_} {code}')
        candidates.append(f'Art. {art} {code}')
        for cand in candidates:
            r, _ = resolve_against_corpus(cand)
            if r: seeds.extend(r); break
    for m in RE_BGE_Q.finditer(query):
        vol, book, page, e = m.groups()
        base = f'BGE {vol} {book} {page}'
        if e:
            r, _ = resolve_against_corpus(f'{base} E. {e}'); seeds.extend(r)
        r, _ = resolve_against_corpus(base); seeds.extend(r)
    for m in RE_BGER_Q.finditer(query):
        case, e = m.groups()
        if e:
            r, _ = resolve_against_corpus(f'{case} E. {e}'); seeds.extend(r)
        r, _ = resolve_against_corpus(case); seeds.extend(r)
    return dedup(seeds)

def resolve_list(cits):
    out = []
    for c in cits:
        r, _ = resolve_against_corpus(c); out.extend(r)
    return out

def predict(query, n=TOP_N_EMIT):
    seeds = extract_seeds(query)
    defaults = resolve_list(UNIVERSAL_DEFAULTS)
    retrieved = [c for c, _ in search_laws(query, top_k=200)]
    merged = dedup(seeds + defaults + retrieved)
    filtered = granularity_filter(merged)
    return filtered[:n]

In [ ]:
import re, csv, json, pickle, time, gc, sys
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

# Cell 8 — Macro F1 + evaluate on val
def per_query_f1(g, p):
    g, p = set(g), set(p)
    if not g and not p: return 1.0
    if not g or not p: return 0.0
    tp = len(g & p)
    if tp == 0: return 0.0
    pr = tp/len(p); rc = tp/len(g)
    return 2*pr*rc/(pr+rc)
def macro_f1(gd, pd_):
    qs = set(gd) | set(pd_)
    return sum(per_query_f1(gd.get(q, []), pd_.get(q, [])) for q in qs) / max(1, len(qs))

val = pd.read_csv(DATA / 'val.csv')
gold_d = {r['query_id']: split_cits(r['gold_citations']) for _, r in val.iterrows()}
pred_d = {r['query_id']: predict(r['query']) for _, r in val.iterrows()}

f1 = macro_f1(gold_d, pred_d)
print(f'\nVAL Macro F1 = {f1:.4f}')
for qid in sorted(pred_d):
    g, p = gold_d[qid], pred_d[qid]
    pf1 = per_query_f1(g, p)
    tp = len(set(g) & set(p))
    print(f'  {qid}: pred={len(p):>2} gold={len(g):>2} tp={tp:>2} F1={pf1:.3f}')

In [ ]:
import re, csv, json, pickle, time, gc, sys
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

# Cell 9 — generate submission.csv
test = pd.read_csv(DATA / 'test.csv')
rows = []
for _, r in test.iterrows():
    cits = predict(r['query'])
    rows.append({'query_id': r['query_id'], 'predicted_citations': ';'.join(cits)})
sub = pd.DataFrame(rows)
out_path = WORK / 'submission.csv'
sub.to_csv(out_path, index=False)
print(f'Wrote {out_path}  rows={len(sub)}')
print(sub.head().to_string(index=False))
counts = sub['predicted_citations'].str.count(';').add(1)
print(f'\nemit count: min={counts.min()} mean={counts.mean():.1f} max={counts.max()}')